In [1]:
import pandas as pd
%run data_preprocessing.ipynb
from collections import Counter


def calculate_journey_length(data, target_column='user_journey', group_by_subscription=False):
    """
    Calculate journey length statistics - number of pages per user journey.
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        group_by_subscription (bool): If True, calculate separately for each subscription type
        
    Returns:
        pd.DataFrame: Statistics about journey lengths
    """
    # Count pages in each journey
    data_copy = data.copy()
    data_copy['journey_length'] = data_copy[target_column].apply(
        lambda x: len(x.split('-')) if pd.notna(x) else 0
    )
    
    if group_by_subscription and 'subscription_type' in data_copy.columns:
        # Group by subscription type
        result = data_copy.groupby('subscription_type')['journey_length'].agg([
            ('avg_length', 'mean'),
            ('min_length', 'min'),
            ('max_length', 'max'),
            ('median_length', 'median'),
            ('total_users', 'count')
        ]).reset_index()
    else:
        # Overall statistics
        stats = {
            'avg_length': data_copy['journey_length'].mean(),
            'min_length': data_copy['journey_length'].min(),
            'max_length': data_copy['journey_length'].max(),
            'median_length': data_copy['journey_length'].median(),
            'total_users': len(data_copy)
        }
        result = pd.DataFrame([stats])
    
    return result


def calculate_page_count(data, target_column='user_journey', group_by_subscription=False):
    """
    Count how many times each page appears across all user journeys.
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        group_by_subscription (bool): If True, calculate separately for each subscription type
        
    Returns:
        pd.DataFrame: Pages sorted by frequency (most common first)
    """
    if group_by_subscription and 'subscription_type' in data.columns:
        # Group by subscription type
        results = []
        
        for sub_type in data['subscription_type'].unique():
            sub_data = data[data['subscription_type'] == sub_type]
            page_counts = Counter()
            
            for journey in sub_data[target_column].dropna():
                pages = journey.split('-')
                page_counts.update(pages)
            
            for page, count in page_counts.items():
                results.append({
                    'subscription_type': sub_type,
                    'page': page,
                    'count': count
                })
        
        result = pd.DataFrame(results)
        result = result.sort_values(['subscription_type', 'count'], ascending=[True, False]).reset_index(drop=True)
    else:
        # Overall count
        page_counts = Counter()
        
        for journey in data[target_column].dropna():
            pages = journey.split('-')
            page_counts.update(pages)
        
        result = pd.DataFrame([
            {'page': page, 'count': count} 
            for page, count in page_counts.items()
        ])
        result = result.sort_values('count', ascending=False).reset_index(drop=True)
    
    return result


def calculate_page_presence(data, target_column='user_journey', group_by_subscription=False):
    """
    Count how many unique journeys contain each page (counts each page once per journey).
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        group_by_subscription (bool): If True, calculate separately for each subscription type
        
    Returns:
        pd.DataFrame: Pages sorted by presence (most common first)
    """
    if group_by_subscription and 'subscription_type' in data.columns:
        # Group by subscription type
        results = []
        
        for sub_type in data['subscription_type'].unique():
            sub_data = data[data['subscription_type'] == sub_type]
            page_presence = Counter()
            
            for journey in sub_data[target_column].dropna():
                unique_pages = set(journey.split('-'))
                page_presence.update(unique_pages)
            
            for page, count in page_presence.items():
                results.append({
                    'subscription_type': sub_type,
                    'page': page,
                    'journey_count': count,
                    'percentage': (count / len(sub_data)) * 100
                })
        
        result = pd.DataFrame(results)
        result = result.sort_values(['subscription_type', 'journey_count'], ascending=[True, False]).reset_index(drop=True)
    else:
        # Overall presence
        page_presence = Counter()
        
        for journey in data[target_column].dropna():
            unique_pages = set(journey.split('-'))
            page_presence.update(unique_pages)
        
        result = pd.DataFrame([
            {'page': page, 'journey_count': count, 'percentage': (count / len(data)) * 100} 
            for page, count in page_presence.items()
        ])
        result = result.sort_values('journey_count', ascending=False).reset_index(drop=True)
    
    return result


def calculate_page_destination(data, target_column='user_journey', top_n=5, group_by_subscription=False):
    """
    Show which pages follow after each page (transition analysis).
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        top_n (int): Number of top destinations to show per page
        group_by_subscription (bool): If True, calculate separately for each subscription type
        
    Returns:
        dict or pd.DataFrame: Page transitions
    """
    if group_by_subscription and 'subscription_type' in data.columns:
        # Group by subscription type - return DataFrame
        results = []
        
        for sub_type in data['subscription_type'].unique():
            sub_data = data[data['subscription_type'] == sub_type]
            transitions = {}
            
            for journey in sub_data[target_column].dropna():
                pages = journey.split('-')
                for i in range(len(pages) - 1):
                    current_page = pages[i]
                    next_page = pages[i + 1]
                    
                    if current_page not in transitions:
                        transitions[current_page] = Counter()
                    transitions[current_page][next_page] += 1
            
            # Get top N for each page
            for page, destinations in transitions.items():
                for dest, count in destinations.most_common(top_n):
                    results.append({
                        'subscription_type': sub_type,
                        'from_page': page,
                        'to_page': dest,
                        'count': count
                    })
        
        result = pd.DataFrame(results)
        result = result.sort_values(['subscription_type', 'from_page', 'count'], 
                                    ascending=[True, True, False]).reset_index(drop=True)
        return result
    else:
        # Overall transitions - return dict
        transitions = {}
        
        for journey in data[target_column].dropna():
            pages = journey.split('-')
            
            for i in range(len(pages) - 1):
                current_page = pages[i]
                next_page = pages[i + 1]
                
                if current_page not in transitions:
                    transitions[current_page] = Counter()
                
                transitions[current_page][next_page] += 1
        
        # Format results - get top N destinations for each page
        result = {}
        for page, destinations in transitions.items():
            top_destinations = destinations.most_common(top_n)
            result[page] = [
                {'destination': dest, 'count': count} 
                for dest, count in top_destinations
            ]
        
        return result


def calculate_page_sequences(data, target_column='user_journey', sequence_length=3, top_n=10, group_by_subscription=False):
    """
    Find the most popular sequences of N consecutive pages.
    
    Args:
        data (pd.DataFrame): Preprocessed DataFrame with grouped user journeys
        target_column (str): Column containing journey strings
        sequence_length (int): Length of sequences to find (default: 3)
        top_n (int): Number of top sequences to return
        group_by_subscription (bool): If True, calculate separately for each subscription type
        
    Returns:
        pd.DataFrame: Top N most common page sequences
    """
    if group_by_subscription and 'subscription_type' in data.columns:
        # Group by subscription type
        results = []
        
        for sub_type in data['subscription_type'].unique():
            sub_data = data[data['subscription_type'] == sub_type]
            sequence_counts = Counter()
            
            for journey in sub_data[target_column].dropna():
                pages = journey.split('-')
                for i in range(len(pages) - sequence_length + 1):
                    sequence = '-'.join(pages[i:i + sequence_length])
                    sequence_counts[sequence] += 1
            
            # Get top N for this subscription type
            for rank, (seq, count) in enumerate(sequence_counts.most_common(top_n), 1):
                results.append({
                    'subscription_type': sub_type,
                    'sequence': seq,
                    'count': count,
                    'rank': rank
                })
        
        result = pd.DataFrame(results)
        result = result.sort_values(['subscription_type', 'rank']).reset_index(drop=True)
    else:
        # Overall sequences
        sequence_counts = Counter()
        
        for journey in data[target_column].dropna():
            pages = journey.split('-')
            for i in range(len(pages) - sequence_length + 1):
                sequence = '-'.join(pages[i:i + sequence_length])
                sequence_counts[sequence] += 1
        
        # Get top N sequences
        top_sequences = sequence_counts.most_common(top_n)
        
        result = pd.DataFrame([
            {'sequence': seq, 'count': count, 'rank': idx + 1} 
            for idx, (seq, count) in enumerate(top_sequences)
        ])
    
    return result


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # Assuming you have loaded grouped_data from preprocessing
    
    print("=" * 80)
    print("METRIC 1: JOURNEY LENGTH")
    print("=" * 80)
    print("\nOverall:")
    length_stats = calculate_journey_length(grouped_data)
    print(length_stats)
    
    print("\nBy Subscription Type:")
    length_by_sub = calculate_journey_length(grouped_data, group_by_subscription=True)
    print(length_by_sub)
    
    print("\n" + "=" * 80)
    print("METRIC 2: PAGE COUNT")
    print("=" * 80)
    print("\nOverall (Top 10):")
    page_counts = calculate_page_count(grouped_data)
    print(page_counts.head(10))
    
    print("\nBy Subscription Type (Top 5 per type):")
    page_counts_by_sub = calculate_page_count(grouped_data, group_by_subscription=True)
    print(page_counts_by_sub.groupby('subscription_type').head(5))
    
    print("\n" + "=" * 80)
    print("METRIC 3: PAGE PRESENCE")
    print("=" * 80)
    print("\nOverall (Top 10):")
    page_presence = calculate_page_presence(grouped_data)
    print(page_presence.head(10))
    
    print("\nBy Subscription Type (Top 5 per type):")
    page_presence_by_sub = calculate_page_presence(grouped_data, group_by_subscription=True)
    print(page_presence_by_sub.groupby('subscription_type').head(5))
    
    print("\n" + "=" * 80)
    print("METRIC 4: PAGE DESTINATION")
    print("=" * 80)
    print("\nOverall (Top 3 destinations for first 5 pages):")
    destinations = calculate_page_destination(grouped_data, top_n=3)
    for i, (page, dests) in enumerate(destinations.items()):
        if i >= 5:
            break
        print(f"\n{page}:")
        for dest in dests:
            print(f"  → {dest['destination']}: {dest['count']} times")
    
    print("\nBy Subscription Type (Sample):")
    destinations_by_sub = calculate_page_destination(grouped_data, top_n=3, group_by_subscription=True)
    print(destinations_by_sub.head(15))
    
    print("\n" + "=" * 80)
    print("METRIC 5: PAGE SEQUENCES (3-page sequences)")
    print("=" * 80)
    print("\nOverall (Top 10):")
    sequences = calculate_page_sequences(grouped_data, sequence_length=3, top_n=10)
    print(sequences)
    
    print("\nBy Subscription Type (Top 5 per type):")
    sequences_by_sub = calculate_page_sequences(grouped_data, sequence_length=3, top_n=5, group_by_subscription=True)
    print(sequences_by_sub)

NameError: name 'cleaned_data' is not defined

NameError: name 'cleaned_data' is not defined